# 03 – Baseline Models: ARIMA, Prophet, XGBoost

This notebook trains and evaluates three baseline forecasting models:
- **ARIMA** – classical statistical time-series model
- **Prophet** – Facebook/Meta time-series model with trend/seasonality decomposition
- **XGBoost** – gradient-boosted trees with engineered features

Predictions are saved for use in notebook 05 (ensemble).

In [1]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import json
from pathlib import Path

from src.utils.data_loader import load_processed_data
from src.utils.metrics import calculate_metrics
from src.visualization.plotter import Plotter
from src.config import RESULTS_DIR

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
plotter = Plotter()

In [2]:
# ── Load processed data ─────────────────────────────────────────────────────
train = load_processed_data('train')
val   = load_processed_data('val')
test  = load_processed_data('test')

print(f'Train: {len(train)} | Val: {len(val)} | Test: {len(test)}')

target_col = 'Close'
y_test = test[target_col]

Train: 1006 | Val: 252 | Test: 249


## A. ARIMA

In [3]:
from src.models.arima_model import ARIMAModel

arima = ARIMAModel(order=(5, 1, 0))
arima.fit(train[target_col])

arima_preds = arima.predict(steps=len(test))
arima_preds.index = test.index[:len(arima_preds)]

arima_metrics = calculate_metrics(y_test.values[:len(arima_preds)], arima_preds.values)
print('ARIMA metrics:', arima_metrics)

arima.save()

ARIMA metrics: {'mse': 37345809.58903244, 'rmse': 6111.121794648871, 'mae': 5714.22688324876, 'mape': 24.622409687370915, 'directional_accuracy': 0.07661290322580645}


/Users/chenshuliu/Desktop/2_2_5152 ADA/2. Project/COMP5152ADA_Project_2/.venv/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/Users/chenshuliu/Desktop/2_2_5152 ADA/2. Project/COMP5152ADA_Project_2/.venv/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/Users/chenshuliu/Desktop/2_2_5152 ADA/2. Project/COMP5152ADA_Project_2/.venv/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/Users/chenshuliu/Desktop/2_2_5152 ADA/2. Project

PosixPath('/Users/chenshuliu/Desktop/2_2_5152 ADA/2. Project/COMP5152ADA_Project_2/models_saved/arima_model.pkl')

## B. Prophet

In [4]:
try:
    from src.models.prophet_model import ProphetModel

    prophet = ProphetModel()
    prophet.fit(pd.concat([train, val]), target_col=target_col)

    prophet_forecast = prophet.predict(periods=len(test))
    prophet_preds = prophet_forecast['yhat'].values[-len(test):]

    prophet_metrics = calculate_metrics(y_test.values, prophet_preds)
    print('Prophet metrics:', prophet_metrics)

    prophet.save()
except Exception as e:
    print(f'Prophet skipped: {e}')
    prophet_preds = None
    prophet_metrics = {}

Importing plotly failed. Interactive plots will not work.
21:04:41 - cmdstanpy - INFO - Chain [1] start processing
21:04:41 - cmdstanpy - INFO - Chain [1] done processing


Prophet metrics: {'mse': 3229197.508968169, 'rmse': 1796.9968027150658, 'mae': 1382.3990855981694, 'mape': 6.629014113713627, 'directional_accuracy': 0.5564516129032258}


## C. XGBoost

In [5]:
from src.models.xgboost_model import XGBoostModel

feature_cols = [c for c in train.columns if c not in [target_col, 'Open', 'High', 'Low', 'Adj Close']]

X_train = train[feature_cols].fillna(0)
y_train_xgb = train[target_col]
X_val   = val[feature_cols].fillna(0)
y_val_xgb = val[target_col]
X_test  = test[feature_cols].fillna(0)

xgb_model = XGBoostModel()
xgb_model.fit(X_train, y_train_xgb, X_val=X_val, y_val=y_val_xgb)

xgb_preds = xgb_model.predict(X_test)
xgb_metrics = calculate_metrics(y_test.values, xgb_preds)
print('XGBoost metrics:', xgb_metrics)

xgb_model.save()

XGBoost metrics: {'mse': 40058619.06441743, 'rmse': 6329.187867682348, 'mae': 5980.781144107681, 'mape': 25.866914562867926, 'directional_accuracy': 0.8508064516129032}


PosixPath('/Users/chenshuliu/Desktop/2_2_5152 ADA/2. Project/COMP5152ADA_Project_2/models_saved/xgboost_model.pkl')

In [6]:
# Feature importance
importance = xgb_model.get_feature_importance()
plotter.plot_feature_importance(importance, top_n=20, filename='feature_importance.png')
print('Feature importance chart saved.')

Feature importance chart saved.


## D. Comparison

In [7]:
all_metrics = {
    'ARIMA':   arima_metrics,
    'XGBoost': xgb_metrics,
}
if prophet_metrics:
    all_metrics['Prophet'] = prophet_metrics

metrics_df = pd.DataFrame(all_metrics).T
print(metrics_df.to_string())
metrics_df.to_csv(RESULTS_DIR / 'baseline_metrics.csv')

                  mse         rmse          mae       mape  directional_accuracy
ARIMA    3.734581e+07  6111.121795  5714.226883  24.622410              0.076613
XGBoost  4.005862e+07  6329.187868  5980.781144  25.866915              0.850806
Prophet  3.229198e+06  1796.996803  1382.399086   6.629014              0.556452


In [8]:
# Predictions comparison plot
predictions = {'ARIMA': arima_preds.values, 'XGBoost': xgb_preds}
if prophet_preds is not None:
    predictions['Prophet'] = prophet_preds

plotter.plot_predictions_comparison(
    y_test.values,
    predictions,
    dates=test.index,
    title='Baseline Models vs Actual (Test Set)',
    filename='baseline_predictions.png'
)
print('Predictions comparison chart saved.')

Predictions comparison chart saved.


In [9]:
# Save predictions for ensemble
preds_df = pd.DataFrame({'actual': y_test.values, 'arima': arima_preds.values, 'xgboost': xgb_preds}, index=test.index)
if prophet_preds is not None:
    preds_df['prophet'] = prophet_preds
preds_df.to_csv(RESULTS_DIR / 'baseline_predictions.csv')
print('Predictions saved to reports/results/baseline_predictions.csv')

Predictions saved to reports/results/baseline_predictions.csv


## Summary

See `reports/results/baseline_metrics.csv` for a full metrics table.

Continue to **04_model_lstm.ipynb** for the deep learning model.